# ERP IDP extraction with MEGaNorm

This tutorial demonstrates **sensor-space ERP imaging-derived phenotypes (IDPs)** with the early `meganorm.src.erp` API.

You will define multiple components, create event-locked epochs, extract amplitude and latency, inspect QC, and prepare a table with **one subject per row and one IDP per column**.

**Run All works without downloading recordings.** The main example generates two synthetic EEG conditions with negative and positive components. Later sections show how to use your own recordings and already-prepared evoked responses.

The component names and windows are examples; configure them for your study. Raw preprocessing currently supports EEG. Prepared-evoked extraction also supports magnetometers and gradiometers.

## 1. Environment and imports

Select a Python ≥3.12 kernel with MEGaNorm and its dependencies installed. When working from this checkout, install it with `python -m pip install -e .` from the repository root using the same environment as your notebook kernel. Jupyter and Matplotlib are needed for this tutorial.

The cell below also supports opening this notebook from either the repository root or the `notebooks` directory. It does not install packages or change your environment.

The synthetic recording helper `meganorm.utils.erp.make_demo_recording` is installed with the package and shared with the tests. Open the notebook from the repository root or its `notebooks/` directory; no tests package is needed.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
repo_root = next((p for p in (cwd, *cwd.parents) if (p / "meganorm" / "src" / "erp.py").is_file()), None)
if repo_root is not None:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from IPython.display import display
from meganorm.src.erp import (
    ERPComponent, ERPConfig, run_erp_pipeline,
    compute_evokeds, extract_erp_idps,
)
from meganorm.utils.erp import make_demo_recording

mne.set_log_level("WARNING")
print(f"Python: {sys.version.split()[0]} | MNE: {mne.__version__}")
from meganorm.plots.plot_erp import plot_erp
from meganorm.src.erp import extract_recording


## 2. Generate a small reproducible recording

Each condition has 20 trials. Both have a negative component at 100 ms. The positive component at Pz is approximately 3 µV for `standard` and 6 µV for `target`, peaking at 300 ms. Small random noise makes the traces more realistic.

MNE stores EEG in **volts** and event samples as **absolute recording sample indices**. This example deliberately uses a nonzero `first_samp` and adds it to the event indices. The original sampling rate is 250 Hz.

In [ ]:
raw, events = make_demo_recording()
event_id = {"standard": 1, "target": 2}
display(pd.DataFrame(events[:6], columns=["absolute_sample", "previous_value", "event_code"]))
print(f"Duration: {raw.n_times / raw.info['sfreq']:.1f} s; channels: {raw.ch_names}")

## 3. Define multiple components

Every entry is measured by the same engine. A component name is a label, not a hard-coded detector.

| Setting | Meaning |
| --- | --- |
| `conditions` | Exact condition names from `event_id` |
| `channels` | Individual sensor names, in output order |
| `window` | Inclusive measurement interval in seconds relative to the event |
| `polarity` | `neg`, `pos`, or `abs`; reported amplitude remains signed |
| `metrics` | Any combination of `peak_amplitude`, `peak_latency`, `mean_amplitude` |
| `sensor_type` | `eeg` by default; determines amplitude units |

Here, N100 is measured at Cz and Pz in both conditions; a custom `late_positive` component is measured at Pz in both conditions. Both share the same epoch and baseline. The current API measures individual sensors, not sensor-group averages.

In [ ]:
components = {
    "N100": ERPComponent(
        conditions=("standard", "target"),
        channels=("Cz", "Pz"),
        window=(0.08, 0.14),
        polarity="neg",
    ),
    "late_positive": ERPComponent(
        conditions=("standard", "target"),
        channels=("Pz",),
        window=(0.2, 0.4),
        polarity="pos",
        metrics=("peak_amplitude", "peak_latency", "mean_amplitude"),
    ),
}

## 4. Configure and run the pipeline

For this controlled synthetic signal, disable ICA, filtering and re-referencing to preserve the known injected amplitudes. **These settings are for this demonstration, not a suggested raw EEG preprocessing protocol.** Baseline correction and automatic peak-to-peak rejection remain enabled.

For real recordings, `ERPConfig()` enables automatic ICA, average reference, and separate ERP/ICA filtering. Section 9 explains that path. Filtering and reference choices can change the measured IDPs.

In [ ]:
demo_config = ERPConfig(
    tmin=-0.2, tmax=0.8,
    baseline=(-0.2, 0.0),
    apply_ica=False,
    reference=None,
    l_freq=None, h_freq=None,
    min_trials=10,
    reject={"eeg": 150e-6},  # peak-to-peak threshold in volts
)

result = run_erp_pipeline(
    raw, "sub-demo-01", components,
    events=events, event_id=event_id,
    config=demo_config,
)
print(f"Retained epochs: {len(result.epochs)}")
print(f"Feature table: {result.features.shape[0]} subject × {result.features.shape[1]} IDPs")

## 5. Inspect evoked waveforms and the extracted values

A condition average is computed once and reused across components. The shaded regions below are the configured measurement windows. Peak amplitude and latency refer to the same sample; a mean amplitude averages all samples inside its window.

In [ ]:
fig, axes = plot_erp(
    result.evokeds, components=components, channels=("Cz", "Pz"),
)
plt.show()

In [ ]:
# This is the actual export structure: one subject per row.
display(result.features)

# Transpose only for easier reading in the notebook.
display(result.features.T.rename(columns={"sub-demo-01": "value"}))

# A quick numerical check against the injected target component.
amplitude_col = "ERP__target__late_positive__peak_amplitude_uV__Pz"
latency_col = "ERP__target__late_positive__peak_latency_ms__Pz"
assert result.features.shape == (1, 14)
assert abs(result.features.loc["sub-demo-01", amplitude_col] - 6.0) < 0.2
assert abs(result.features.loc["sub-demo-01", latency_col] - 300.0) <= 12.0
print("Synthetic amplitude and latency checks passed.")

## 6. Inspect quality control

`result.qc` is separate from the IDP table. It contains a status and trial counts for each IDP. Missing or bad sensors, missing conditions and too few retained trials produce `NaN` with a reason. Peak metrics also become unavailable if the requested polarity is absent.

Invalid windows and sensor-type mismatches raise errors. Epoch rejection is based on configured thresholds and existing BAD annotations; AutoReject interpolation is not yet integrated into this path.

In [ ]:
display(result.qc)
display(result.qc[["condition", "n_events", "n_trials", "n_rejected"]].drop_duplicates())
print("ICA applied:", result.preprocessing_qc["ica_applied"])
print("Excluded ICA components:", result.preprocessing_qc["ica_excluded"])
print("Epoch drop log:", result.preprocessing_qc["drop_log"])
assert result.qc["status"].eq("ok").all()

### An unavailable measurement keeps its column

The optional component below requests an absent sensor. This illustrates stable feature columns and QC reporting without changing the original component definitions.

In [ ]:
missing_sensor_component = {
    "custom": ERPComponent(
        conditions=("target",), channels=("Fz",),
        window=(0.2, 0.4), polarity="pos",
    )
}
unavailable_features, unavailable_qc = extract_erp_idps(
    "sub-demo-01", result.evokeds, missing_sensor_component, min_trials=10,
)
display(unavailable_features)
display(unavailable_qc)
assert unavailable_features.isna().all().all()
assert unavailable_qc["status"].eq("missing_channel").all()

## 7. Combine subjects and optionally save results

Process each subject using the same component definitions, then concatenate rows. Use unique subject identifiers. QC belongs in a separate file and should not automatically be selected as a normative-model response.

Saving is **off by default**. Enable it to write synthetic demonstration results under `results/erp_tutorial/`. This separate directory avoids replacing spectral subject rows with ERP-only results.

In [ ]:
raw_2, events_2 = make_demo_recording(seed=43)
result_2 = run_erp_pipeline(
    raw_2, "sub-demo-02", components,
    events=events_2, event_id=event_id, config=demo_config,
)
results = [result, result_2]
all_features = pd.concat([item.features for item in results])
all_qc = pd.concat([item.qc for item in results], ignore_index=True)
assert all_features.index.is_unique
assert all_features.shape == (2, 14)
display(all_features)

SAVE_OUTPUTS = False
output_dir = (repo_root or cwd) / "results" / "erp_tutorial"
if SAVE_OUTPUTS:
    output_dir.mkdir(parents=True, exist_ok=True)
    all_features.to_csv(output_dir / "all_features.csv")
    all_qc.to_csv(output_dir / "erp_qc.csv", index=False)
    for item in results:
        subject_id = item.features.index[0]
        item.features.to_csv(output_dir / f"{subject_id}.csv")
    print(f"Saved synthetic results to {output_dir}")
else:
    print("Saving disabled; set SAVE_OUTPUTS=True to export these tables.")

## 8. Use already-prepared epochs or evoked responses

Use `compute_evokeds(epochs)` to average existing event-locked epochs, or pass a dictionary of prepared evoked responses directly to `extract_erp_idps()`. This bypasses preprocessing and allows reuse of an established cleaning workflow.

`nave` is interpreted as the retained trial count: use ordinary condition averages with accurate counts. Weighted contrasts with effective sample sizes are not supported by this trial-count rule.

In [ ]:
prepared_evokeds = compute_evokeds(result.epochs)
prepared_features, prepared_qc = extract_erp_idps(
    "sub-demo-01", prepared_evokeds, components, min_trials=10,
)
pd.testing.assert_frame_equal(prepared_features, result.features)
print("Prepared-epoch extraction matches the raw-pipeline IDPs.")

For prepared MEG evoked responses, define components with `sensor_type="mag"` or `"grad"` and the matching sensor names. Amplitude units are T for magnetometers and T/m for gradiometers; latency remains ms. This does not enable MEG raw preprocessing in `run_erp_pipeline()`.

```python
meg_components = {
    "auditory_field": ERPComponent(
        conditions=("auditory",), channels=("MEG0111",),
        window=(0.08, 0.15), polarity="abs", sensor_type="mag",
    )
}
# features, qc = extract_erp_idps(
#     "sub-01", {"auditory": auditory_evoked}, meg_components, min_trials=10,
# )
```

## 9. Apply automatic preprocessing to your EEG recordings

The package wrapper `meganorm.src.erp.extract_recording()` loads the recording and calls the ERP pipeline. The example below is **not called by Run All**. Supply a recording path and the exact annotation descriptions for your experiment. It uses automatic preprocessing rather than the synthetic settings.

- Set correct channel types and sensor locations before processing. The existing `prepare_eeg_data()` helper can read metadata when the dataset follows its conventions.
- ICA fits on a separately filtered copy and applies exclusions to the ERP-filtered data. It reuses the package's correlation helper for available EOG/ECG channels. With no usable auxiliary channels, it uses ICLabel and requires EEG locations.
- The defaults shown are starting protocol settings, not validated settings for every component. Choose reference, filters, rejection threshold and minimum trials for your study.
- There are no interactive ICA or epoch-selection steps. Review the saved QC before interpreting results.

In [ ]:
# extract_recording is imported from meganorm.src.erp above.
# Adapt and uncomment to process your own recording:
# real_result = extract_recording(
#     "path/to/sub-01_task-oddball_eeg.vhdr", "sub-01", components,
#     annotation_map={"standard": "Stimulus/S  5", "target": "Stimulus/S  6"},
#     config=ERPConfig(
#         tmin=-0.2, tmax=0.8, baseline=(-0.2, 0.0),
#         apply_ica=True, reference="average", min_trials=10,
#     ),
# )
# real_result.features.to_csv("sub-01_erp.csv")
# real_result.qc.to_csv("sub-01_erp_qc.csv", index=False)

### Event inputs and resampling

Use **one** of these input forms:

1. `annotation_map={condition: exact_annotation_description}` for events stored as annotations.
2. `events=events, event_id={condition: integer_code}` for an explicit MNE event array, for example from a known trigger channel.

The event array must use the original recording's sample rate and include `raw.first_samp`. Events are resolved before cleaning. Set `ERPConfig(resample_sfreq=...)` to resample data and events together; do not independently resample the event indices. Repeated sample indices fail explicitly, including collisions created by downsampling. BIDS `events.tsv` reading is not currently built into this API.

## 10. Next steps and limits

Use `all_features` as the subject-by-IDP table when joining demographic data and selecting normative-model response columns. Component windows, units, preprocessing and sensor definitions should be consistent across subjects and sites.

This early implementation does not yet add SLURM dispatch, source reconstruction, automatic bad-channel detection, AutoReject interpolation, sensor-group averaging, bootstrap intervals or multi-session aggregation. The synthetic checks demonstrate API behavior and numerical extraction, not scientific validation on real recordings.

Further details: [ERP quickstart](../docs/erp_quickstart.md).